# Chapter 4 — inside the network: initialization, dead neurons, BatchNorm

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 4 — inside the network: initialization, dead neurons, BatchNorm

**Video:** 1h55m · [youtu.be/P6sfmUTpUmc](https://youtu.be/P6sfmUTpUmc) · **The most practically useful lecture in the course**, and the most underrated. [my read]

### The problem

The Chapter 3 network trains, but badly, for reasons invisible in the loss curve. This chapter opens the machine and looks at the numbers moving through it.

> **Say it to a six-year-old.** Imagine shouting a message down a line of a hundred friends. If everyone shouts a bit louder than they heard, by the end it is unbearable noise. If everyone whispers a bit quieter, by the end there is silence and the message is lost. You want everyone to pass it on at about the same volume they heard it. That is this entire chapter: keeping the volume steady all the way down the line.

### Bug 1: confidently wrong at birth

**Run it.** Measure the very first loss before any training:

In [ ]:
# using the Chapter 3 network exactly as built
ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
emb = C[Xtr[ix]]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
import math
print("initial loss:", round(F.cross_entropy(logits, Ytr[ix]).item(), 4))
print("what it should be:", round(math.log(27), 4))
print("logit spread (max - min):", round((logits.max() - logits.min()).item(), 2))

**What you should see:**

**Expected output:**

```
initial loss: 27.8817
what it should be: 3.2958
logit spread (max - min): 70.44
```

[verified]

The logits are spread across a range of 70, so after softmax (which exponentiates, section 1.4) the model is essentially certain about one arbitrary character, and it is wrong most of the time. The first few hundred training steps do nothing but squash that unearned confidence.

**The fix:** scale the final layer's weights down at initialization.

In [ ]:
W2 = torch.randn((n_hidden, 27), generator=g) * 0.01   # was * 1.0
b2 = torch.zeros(27)                                   # bias can start at exactly 0

**Why 0.01 and not 0.** Because identical neurons receive identical gradients forever and never differentiate, a problem called **symmetry breaking**. Karpathy keeps it "not exactly zero, it's got some little entropy" [transcript]. The biases *can* be exactly zero, since they are already distinct by virtue of feeding different neurons.

### Bug 2: dead neurons

**Run it.** Look at the hidden layer's activations:

In [ ]:
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
saturated = (h.abs() > 0.99).float().mean().item()
print(f"fraction of hidden units pinned at ±1: {saturated*100:.1f}%")
print("a few activation values:", [round(v, 3) for v in h[0, :8].tolist()])

**What you should see** (with the naive initialization):

**Expected output:**

```
fraction of hidden units pinned at ±1: 61.0%
a few activation values: [0.81, -0.9, -0.999, 0.998, -0.651, -0.69, -0.962, -1.0]
```

[verified]

61% of the layer is jammed at the extremes of `tanh`. Recall from section 1.10 that the slope there is 0.0013, effectively zero. **Gradient cannot flow through those neurons, so they barely train.** The network has 200 hidden units and is effectively using a fraction of them.

A neuron that is saturated for *every* example in the batch is dead in the strict sense and will likely never recover.

**Visualize it.** Karpathy plots this as a black-and-white image, 32 examples by 200 neurons, white marking saturation:

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(16, 4))
plt.imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest')
plt.xlabel('neuron'); plt.ylabel('example')
plt.show()

Large white regions mean wasted capacity. A fully white *column* is a dead neuron.

### The fix: principled initialization

Too large and activations saturate. Too small and the signal shrinks toward zero as it passes through layers until nothing is left. The right scale keeps the standard deviation of activations roughly constant from layer to layer.

**Standard deviation**, if the term is new: a measure of spread. A standard deviation of 1 means typical values sit about 1 away from the average. Doubling every number doubles it.

**The rule** (He et al. 2015, **Kaiming initialization**): scale initial weights by `gain / sqrt(fan_in)`, where `fan_in` is the number of inputs feeding each neuron and `gain` compensates for the nonlinearity, `5/3` for `tanh`. [standard]

**Why divide by √fan_in.** Each neuron sums `fan_in` independent random products. When you add up n independent random numbers, the spread grows as √n. So dividing the weights by √fan_in cancels exactly that growth and holds the output spread at 1, no matter how wide the layer.

**Run it.** Confirm the rule empirically rather than trusting it:

In [ ]:
import torch
x = torch.randn(1000, 30)                              # 1000 examples, 30 inputs
for scale, name in [(1.0, 'naive'), ((5/3)/30**0.5, 'kaiming')]:
    W = torch.randn(30, 200) * scale
    h = torch.tanh(x @ W)
    print(f"{name:8} weight std {scale:.4f} -> activation std {h.std():.4f}, "
          f"saturated {(h.abs() > 0.99).float().mean()*100:.1f}%")

**What you should see:**

**Expected output:**

```
naive    weight std 1.0000 -> activation std 0.9225, saturated 61.5%
kaiming  weight std 0.3043 -> activation std 0.7547, saturated 11.0%
```

[verified]

61.5% saturated versus 11.0%, from one multiplication. Karpathy's target is roughly 5% [transcript]; this single-layer test lands at 11% because the `5/3` gain is deliberately generous, trading a little extra saturation for a stronger signal reaching deeper layers. Some saturation is fine, most is not.

**The effect on real training:**

| Setup | Initial loss | Saturated at init | Train | Dev |
|---|---|---|---|---|
| Naive init | 27.8817 | 61.0% | 2.2618 | 2.2778 |
| Fixed init | 3.3179 | 8.1% | 2.1178 | **2.1481** |
| Fixed init + BatchNorm | 3.3147 | 0.5% | 2.1508 | 2.1653 |

All [verified], 30,000 steps each, identical seeds and hyperparameters.

The fix is worth about 0.13 of loss, which at this scale is a large improvement, and it costs one line. Note also that the initial loss went from 27.88 to 3.3179, within a whisker of the theoretical 3.2958.

**An honest note about that table:** BatchNorm came out slightly *worse* than plain fixed initialization here (2.1653 versus 2.1481). At this small scale that is expected, and the lecture makes the same point: once initialization is correct, BatchNorm's value is in making deep networks trainable at all, not in squeezing the last decimal from a 2-layer one. Reporting it the other way round would have been tidier and false. [verified]

### BatchNorm

**Batch Normalization** (Ioffe and Szegedy, 2015) attacks the same problem from the other end: rather than choosing weights so activations come out well-behaved, force them to be well-behaved.

**The mechanism:**

1. Take the pre-activation values for the batch: shape 32×200, meaning 32 examples by 200 neurons. [transcript]
2. For each neuron, compute the mean and standard deviation **across the 32 examples**.
3. Subtract the mean, divide by the standard deviation. Each neuron's output across the batch now has mean 0 and standard deviation 1.
4. Multiply by a learned `gain` and add a learned `bias`, both trainable, so the network can undo the normalization if that helps.

**Run it.**

In [ ]:
hpreact = emb.view(-1, 30) @ W1 + b1
bngain, bnbias = torch.ones((1, 200)), torch.zeros((1, 200))
normalized = bngain * (hpreact - hpreact.mean(0, keepdim=True)) / hpreact.std(0, keepdim=True) + bnbias
print("before: mean %.3f std %.3f" % (hpreact.mean().item(), hpreact.std().item()))
print("after:  mean %.3f std %.3f" % (normalized.mean().item(), normalized.std().item()))

**What you should see:**

**Expected output:**

```
before: mean 0.347 std 5.181
after:  mean 0.000 std 0.984
```

[verified]

The mean is exactly 0 and the standard deviation is 0.984 rather than exactly 1.000. That is not an error: `.std()` in PyTorch divides by `n−1` rather than `n` (Bessel's correction, the standard unbiased estimator), so normalizing by it leaves the measured spread very slightly under 1. With a batch of 32 the discrepancy is about 1.6%, and it shrinks as batches grow.

Note `.mean(0)`: dimension 0 is the batch dimension, so this averages **across examples**, one statistic per neuron. Using `.mean(1)` instead would average across neurons within each example, which is a different operation entirely and is in fact LayerNorm (Chapter 7).

**Analogy.** Grading on a curve. Whatever the raw scores, the class ends up with a fixed average and spread, so the next stage always receives input on a predictable scale.

**Why it mattered historically.** It made deep networks trainable without painstaking per-layer tuning. Karpathy: "batch normalization was very influential at the time when it came out in roughly 2015 because it was kind of the first time that you could train reliably much deeper neural nets" [transcript].

**Its ugly side, stated plainly.** BatchNorm couples the examples in a batch: your prediction for one name now depends on which other names happened to share its batch. That is mathematically unpleasant and causes a long tail of bugs:

- **Running statistics.** At test time you may have a single example and no batch to average over, so training must maintain a running mean and variance on the side, updated with momentum around 0.001 to 0.1, used at inference. [transcript]
- **The preceding bias becomes pointless.** Subtracting the batch mean cancels any bias added just before, so that bias is a no-op consuming memory and gradient.
- **Two modes.** The layer behaves differently in training and evaluation, so you must flip a flag, and forgetting is a classic silent bug.

**Run it.** The full version with running statistics:

In [ ]:
bnmean_running, bnstd_running = torch.zeros((1, 200)), torch.ones((1, 200))

# --- during training
bnmeani = hpreact.mean(0, keepdim=True)
bnstdi  = hpreact.std(0, keepdim=True)
hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
with torch.no_grad():
    bnmean_running = 0.999*bnmean_running + 0.001*bnmeani
    bnstd_running  = 0.999*bnstd_running  + 0.001*bnstdi

# --- at evaluation time, use the running estimates instead
# hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias

Karpathy's verdict is that people use BatchNorm because it works, while quietly wishing they did not have to. Chapter 7 introduces the successor, **LayerNorm**, which normalizes across the features of a single example and removes the coupling entirely. Transformers use LayerNorm.

> **For the PhD in the room.** The original paper motivated BatchNorm as reducing "internal covariate shift," and that explanation has not held up: Santurkar et al. (2018) showed you can inject noise after BatchNorm, restoring covariate shift, and keep the benefit, arguing instead that it smooths the loss landscape and permits larger stable learning rates. Also worth knowing: BatchNorm's regularization effect comes from the batch-dependent noise in the statistics, which is why it interacts badly with small batches, and why the train/eval discrepancy is a genuine distribution shift rather than an implementation detail. The modern transformer stack has largely abandoned it in favour of LayerNorm and RMSNorm, the latter dropping the mean subtraction entirely on the grounds that only the rescaling was load-bearing.

### The diagnostic toolkit

The lasting value of this lecture is four plots to make when a network misbehaves. [transcript]

1. **Activation histograms per layer.** Look for saturation. Target roughly 5%; watch for it growing or collapsing with depth.
2. **Gradient histograms per layer.** Look for uniformity. In one run the last layer's gradients were about 10× larger than every other layer's, meaning that layer trained 10× faster than the rest, an imbalance invisible in the loss curve.
3. **Weight gradient distributions.** The same idea, per parameter tensor.
4. **Update-to-data ratio over time.** For each parameter tensor, compare the size of the update to the size of the parameter itself.

**Run it.** The fourth one, which catches the most real problems: [my read]

In [ ]:
with torch.no_grad():
    for p, name in zip(parameters, ['C', 'W1', 'b1', 'W2', 'b2']):
        ratio = ((0.1 * p.grad).std() / p.data.std()).log10().item()
        print(f"{name:3} log10(update/data) = {ratio:+.2f}")

**What you should see** (roughly; yours will vary):

**Expected output:**

```
C   log10(update/data) = -1.39
W1  log10(update/data) = -2.10
b1  log10(update/data) = -1.98
W2  log10(update/data) = -2.26
b2  log10(update/data) = -2.12
```

[verified, measured at initialization on the naive network]

**The rule of thumb: this number should sit near −3**, meaning updates are about 1/1000th of the parameter magnitude. [transcript] Much higher, say −1, and that tensor is being trained too aggressively; much lower, say −5, and it is effectively frozen.

Every number above is too high, and `C` at −1.39 is the worst: the embedding table is being rewritten by about 4% of its own size on every single step. That is this chapter's broken network, caught by a diagnostic rather than by a loss curve. Re-run the same three lines after fixing the initialization and the values settle toward −3. **This is the plot to reach for first when a network trains badly for no visible reason.** [my read]

### Exercises

1. **Reproduce the table.** Train the same network three ways, naive, fixed init, and fixed init plus BatchNorm, and confirm you get the initial losses 27.88, 3.32, 3.31.
2. **Sweep the initialization scale** on `W1` from 0.01 to 3.0 and plot final dev loss against it. There is a broad valley, not a knife edge.
3. **Break BatchNorm on purpose** by evaluating with batch statistics instead of running statistics on a single example, and watch the prediction change depending on what else is in the batch.
4. **Remove `tanh` entirely** and retrain. The loss barely changes, because a 2-layer network is nearly linear anyway. Then add three more layers with and without `tanh`, and the difference appears.

### Troubleshooting

| Symptom | Cause |
|---|---|
| Initial loss is huge (20+) | Output layer weights too large; multiply by 0.01 |
| Loss drops fast then flatlines high | Saturated neurons; check the `abs() > 0.99` fraction |
| Model fine in training, broken at eval | BatchNorm using batch statistics at evaluation time; use running statistics |
| Dev loss oscillates wildly between evaluations | Batch too small for BatchNorm statistics to be stable; raise it |
| `RuntimeError: running_mean should contain 200 elements` | Shape mismatch between the BatchNorm buffers and the layer width |

### 30-second version

Before touching the architecture, look at the numbers inside the network. If the first loss is 27.88 when arithmetic says 3.2958, the initialization is broken. If 61% of neurons are pinned at the extremes of their squashing function, they are dead and cannot learn. Scaling the starting weights by `gain/√fan_in` fixes both and is worth 0.13 of loss for one line of code. BatchNorm forces each layer's output onto a fixed scale instead, which is what made deep networks trainable in 2015, at the cost of coupling every example to whatever else is in its batch.

---